# Resource Agent - Resource-Sizing Model (Kaggle Training)

Trains the **Resource Agent's** supervised regression model, which recommends the best
compute **settings** for each pipeline stage - `workers`, `DIU`, `peak memory`,
`shuffle partitions`, and `node type` - always within the student-tier hard limits.

This notebook is **self-contained**: it generates the 500,000-row training set and trains
the models in one place. It reproduces `resource_agent/ml/feature_spec.py`,
`ml/calibration.py`, and `training/` from the repo so the model it produces is
**train/serve compatible** with the FastAPI agent.

**Pipeline:** generate 500k rows -> train 4 regressors + 1 node classifier -> save resource_models.pkl.

### Why a model at all?
Runtime/SLA prediction is owned by the **Performance Prediction Agent**. The Resource Agent
owns *resource management*: given a stage + its data, pick the cheapest settings that still
fit. That is a tabular regression problem - classic ML, not an LLM.

### Grounding in real data
Labels come from a demand policy calibrated to the real telemetry in `Datasets/Cleaned/`
(job_runs = 779k Databricks durations, pipeline_runs = ADF copies, queries = op-cost
ordering group_by>join>agg, dbquery_statistics = CPU/IO magnitudes). If you upload those
CSVs as a Kaggle dataset, the generator reads the real trigger mix from them; otherwise it
falls back to the measured distribution baked in below.

## 0. Pin scikit-learn to match the serving environment

**Critical:** joblib pickles are tied to the scikit-learn version that wrote them.
The FastAPI app runs **scikit-learn 1.7.0**, so we pin the same version here - otherwise
the agent fails to load the bundle and silently falls back to the heuristic.
After this cell runs, **restart the kernel** if Kaggle prompts you to, then run the rest.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn==1.7.0", "joblib>=1.3", "pandas", "numpy"], check=True)
import sklearn
print("scikit-learn:", sklearn.__version__)   # must print 1.7.0

## 1. Shared contract - features & targets

Mirrors `resource_agent/ml/feature_spec.py` **exactly**. Keep this in sync with the repo:
the runtime predictor builds its feature vector from the same `FEATURE_COLS` in the same order.

In [ ]:
# Student-tier hard limits + node catalogue (mirror resource_agent.py)
MAX_WORKERS, MAX_DIU, MAX_CONCURRENT, MAX_TOTAL_MEM_GB = 4, 8, 3, 64.0
NODE_SPECS = {
    "Standard_D4s_v3":  {"cpu": 4, "memory_gb": 16.0},
    "Standard_D4_v3":   {"cpu": 4, "memory_gb": 16.0},
    "Standard_DS3_v2":  {"cpu": 4, "memory_gb": 14.0},
    "Standard_DS4_v2":  {"cpu": 8, "memory_gb": 28.0},
    "Standard_DS2_v2":  {"cpu": 2, "memory_gb": 7.0},
    "Standard_D8s_v3":  {"cpu": 8, "memory_gb": 32.0},
}
DEFAULT_NODE = "Standard_D4s_v3"
NODE_TYPES_BY_MEM = sorted(NODE_SPECS, key=lambda n: NODE_SPECS[n]["memory_gb"])

FEATURE_COLS = [
    "stage_is_copy", "csv_size_mb", "row_count", "column_count", "size_hint_ord",
    "transform_count", "agg_count", "has_filter", "has_join", "has_groupby",
    "has_aggregation", "has_distinct", "has_sort", "stage_index", "n_stages",
    "is_final_stage",
]
TARGET_COLS = ["rec_workers", "rec_diu", "rec_memory_gb", "rec_shuffle_partitions"]
NODE_TARGET = "rec_node_type"
SHUFFLE_TIERS = [8, 16, 32, 64, 128, 200]
SIZE_HINT_LABELS = ["small", "medium", "large", "xlarge"]

def size_hint_to_ord(size_hint, csv_size_mb=0.0):
    s = (size_hint or "").lower()
    for i, label in enumerate(SIZE_HINT_LABELS):
        if label in s:
            return i
    return 0 if csv_size_mb < 5 else 1 if csv_size_mb < 50 else 2 if csv_size_mb < 200 else 3

def snap_shuffle(v):
    return min(SHUFFLE_TIERS, key=lambda t: abs(t - v))

def stage_features(stage, schema, csv_size_bytes=0, stage_index=0, n_stages=1):
    schema = schema or {}
    is_copy = 1 if (stage.get("type") or "notebook").lower() == "copy" else 0
    csv_size_mb = (csv_size_bytes / (1024*1024)) if csv_size_bytes else 0.0
    row_count = int(schema.get("row_count", 0) or 0)
    cols = schema.get("columns") or list((schema.get("inferred_types") or {}).keys())
    transforms = stage.get("transformations") or []
    tjoined = " ".join(str(t) for t in transforms).lower()
    agg = stage.get("aggregations") or stage.get("aggregation") or {}
    agg = agg if isinstance(agg, dict) else {}
    exprs = agg.get("agg_exprs") or agg.get("aggregations") or []
    agg_count = len(exprs) if isinstance(exprs, list) else 0
    gb = agg.get("group_by") or agg.get("groupBy") or []
    return {
        "stage_is_copy": is_copy, "csv_size_mb": round(csv_size_mb, 6),
        "row_count": row_count, "column_count": len(cols),
        "size_hint_ord": size_hint_to_ord(schema.get("size_hint", ""), csv_size_mb),
        "transform_count": len(transforms), "agg_count": agg_count,
        "has_filter": 1 if stage.get("filter_condition") else 0,
        "has_join": 1 if "join" in tjoined else 0,
        "has_groupby": 1 if gb else 0,
        "has_aggregation": 1 if (agg_count > 0 or agg) else 0,
        "has_distinct": 1 if "distinct" in tjoined else 0,
        "has_sort": 1 if ("orderby" in tjoined or "sort" in tjoined) else 0,
        "stage_index": int(stage_index), "n_stages": int(n_stages),
        "is_final_stage": 1 if stage_index >= n_stages - 1 else 0,
    }
print("contract ready:", len(FEATURE_COLS), "features,", len(TARGET_COLS)+1, "targets")

## 2. Calibration - the demand->settings label policy

Mirrors `resource_agent/ml/calibration.py`. Constants are anchored to the real telemetry
(see the markdown at the top). This function labels every training row, so the learned model
imitates a **demand-driven provisioning policy**, not the old duration heuristic.

In [ ]:
import math

ROWS_PER_CORE_PER_S, ADF_MB_PER_DIU_PER_S = 1600.0, 5.0
TARGET_COPY_S, SOFT_TARGET_S, DRIVER_ONLY_ROWS = 55.0, 120.0, 60_000
BYTES_PER_ROW, ROWS_PER_MB = 140.0, 1100.0
DRIVER_OVERHEAD_GB, MEM_PARTITION_MB, USABLE_MEM_FRACTION = 4.0, 128.0, 0.6

WORK = {"transform_per":0.08,"filter":0.05,"agg_per":0.90,"join":2.00,"groupby":1.60,"distinct":1.20,"sort":0.80}
SHUFFLE_BLOWUP = {"base":1.0,"transform_per":0.05,"agg_per":0.25,"join":0.80,"groupby":0.60,"distinct":0.30}

_DEFAULT_MEM = NODE_SPECS[DEFAULT_NODE]["memory_gb"]
NODE_CANDIDATES = [n for n in NODE_TYPES_BY_MEM
                   if NODE_SPECS[n]["cpu"] >= 4 and NODE_SPECS[n]["memory_gb"] >= _DEFAULT_MEM]

def _work_multiplier(f):
    return (1.0 + WORK["transform_per"]*f["transform_count"] + WORK["filter"]*f["has_filter"]
            + WORK["agg_per"]*f["agg_count"] + WORK["join"]*f["has_join"]
            + WORK["groupby"]*f["has_groupby"] + WORK["distinct"]*f["has_distinct"] + WORK["sort"]*f["has_sort"])

def _shuffle_blowup(f):
    return (SHUFFLE_BLOWUP["base"] + SHUFFLE_BLOWUP["transform_per"]*f["transform_count"]
            + SHUFFLE_BLOWUP["agg_per"]*f["agg_count"] + SHUFFLE_BLOWUP["join"]*f["has_join"]
            + SHUFFLE_BLOWUP["groupby"]*f["has_groupby"] + SHUFFLE_BLOWUP["distinct"]*f["has_distinct"])

def _data_gb(f):
    mb = f["csv_size_mb"] or (f["row_count"]*BYTES_PER_ROW/(1024*1024) if f["row_count"] else 0)
    return mb/1024.0

def _effective_rows(f):
    rows = f["row_count"] or (f["csv_size_mb"]*ROWS_PER_MB)
    return rows*_work_multiplier(f)

def _pick_node(per_worker_gb):
    for n in NODE_CANDIDATES:
        if NODE_SPECS[n]["memory_gb"]*USABLE_MEM_FRACTION >= per_worker_gb:
            return n
    return NODE_CANDIDATES[-1]

def _noise(rng, sigma):
    return math.exp(rng.gauss(0.0, sigma))

def label_settings(f, rng):
    if f["stage_is_copy"]:
        raw = f["csv_size_mb"]/max(TARGET_COPY_S*ADF_MB_PER_DIU_PER_S, 1e-6)
        diu = int(min(MAX_DIU, max(2, math.ceil(raw*_noise(rng, 0.12)))))
        return {"rec_workers":0,"rec_diu":diu,"rec_memory_gb":round(diu*1.5,2),
                "rec_shuffle_partitions":SHUFFLE_TIERS[0],"rec_node_type":DEFAULT_NODE}
    node_cores = NODE_SPECS[DEFAULT_NODE]["cpu"]
    eff_rows = _effective_rows(f)
    working_gb = _data_gb(f)*_shuffle_blowup(f)*_noise(rng, 0.15)
    parallel_w = MAX_WORKERS
    for w in range(1, MAX_WORKERS+1):
        if eff_rows/(w*node_cores*ROWS_PER_CORE_PER_S) <= SOFT_TARGET_S:
            parallel_w = w; break
    biggest = NODE_SPECS[NODE_TYPES_BY_MEM[-1]]["memory_gb"]*USABLE_MEM_FRACTION
    memory_w = max(1, math.ceil(working_gb/max(biggest, 1e-6)))
    workers = int(min(MAX_WORKERS, max(parallel_w, memory_w)))
    driver_cap = NODE_SPECS[DEFAULT_NODE]["memory_gb"]*USABLE_MEM_FRACTION
    no_heavy = not (f["has_aggregation"] or f["has_join"] or f["has_groupby"])
    if eff_rows < DRIVER_ONLY_ROWS and working_gb <= driver_cap and no_heavy:
        workers = 0
    node = _pick_node(working_gb/max(workers, 1))
    mem_gb = min(MAX_TOTAL_MEM_GB, round(DRIVER_OVERHEAD_GB+working_gb, 2))
    shuffle_raw = (working_gb*1024.0/MEM_PARTITION_MB)*(1.0+0.5*f["has_groupby"]+0.3*f["has_join"])
    shuffle = snap_shuffle(max(SHUFFLE_TIERS[0], shuffle_raw*_noise(rng, 0.1)))
    return {"rec_workers":workers,"rec_diu":0,"rec_memory_gb":mem_gb,
            "rec_shuffle_partitions":shuffle,"rec_node_type":node}
print("calibration ready")

## 3. Generate 500,000 training rows

Each row is one pipeline **stage**. Optionally reads the real trigger mix from an uploaded
`job_runs_cleaned.csv` (attach the cleaned CSVs as a Kaggle dataset under `/kaggle/input/`).

In [ ]:
import random, glob, pandas as pd

N_ROWS = 500_000
SEED = 20260707
SIZE_TIERS = {"small":(0.30,(500,49_000)),"medium":(0.30,(55_000,980_000)),
              "large":(0.20,(1_100_000,9_500_000)),"xlarge":(0.20,(11_000_000,120_000_000))}
N_STAGES_DIST = {2:0.20,3:0.30,4:0.25,5:0.15,6:0.10}
TRANSFORM_WEIGHTS = {0:0.10,1:0.20,2:0.22,3:0.18,4:0.12,5:0.08,6:0.05,7:0.03,8:0.02}
FINAL_AGG_PROB, FILTER_PROB, JOIN_PROB, DISTINCT_PROB, SORT_PROB = 0.35,0.40,0.08,0.10,0.10
COLUMN_RANGE = (5, 28)
DEFAULT_TRIGGER_MIX = {"CONTINUOUS":0.828,"CRON":0.160,"ONETIME":0.009,"PERIODIC":0.003}
TRIGGER_INTENSITY = {"CONTINUOUS":1.0,"CRON":1.6,"ONETIME":1.4,"PERIODIC":1.3}

def _weighted(rng, dist):
    return rng.choices(list(dist), weights=list(dist.values()), k=1)[0]

def load_trigger_mix():
    hits = glob.glob("/kaggle/input/**/job_runs_cleaned.csv", recursive=True)
    if not hits:
        print("[gen] no job_runs_cleaned.csv in /kaggle/input - using measured trigger mix")
        return DEFAULT_TRIGGER_MIX
    try:
        s = pd.read_csv(hits[0], usecols=["trigger_type"])["trigger_type"].value_counts(normalize=True)
        mix = {k: float(v) for k, v in s.items() if k in TRIGGER_INTENSITY}
        tot = sum(mix.values())
        print("[gen] real trigger mix from", hits[0])
        return {k: v/tot for k, v in mix.items()} if tot else DEFAULT_TRIGGER_MIX
    except Exception as e:
        print("[gen] read failed:", e); return DEFAULT_TRIGGER_MIX

def make_notebook_stage(rng, is_final):
    transforms = ["col%d = expr%d" % (i, i) for i in range(_weighted(rng, TRANSFORM_WEIGHTS))]
    stage = {"type":"notebook","transformations":transforms}
    if is_final and rng.random() < FINAL_AGG_PROB:
        stage["aggregations"] = {"group_by":["grp"],"agg_exprs":["sum(c%d)" % i for i in range(rng.randint(1,3))]}
    if rng.random() < FILTER_PROB: stage["filter_condition"] = "x > 0"
    if rng.random() < JOIN_PROB: transforms.append("j = join(other)")
    if rng.random() < DISTINCT_PROB: transforms.append("distinct()")
    if rng.random() < SORT_PROB: transforms.append("orderBy(x)")
    return stage

def generate(n_rows, seed):
    rng = random.Random(seed)
    trigger_mix = load_trigger_mix()
    rows = []
    while len(rows) < n_rows:
        tier = _weighted(rng, {k:v[0] for k,v in SIZE_TIERS.items()})
        lo, hi = SIZE_TIERS[tier][1]
        row_count = rng.randint(lo, hi)
        csv_size_bytes = int(row_count*max(40.0, rng.gauss(BYTES_PER_ROW, 40.0)))
        intensity = TRIGGER_INTENSITY[_weighted(rng, trigger_mix)]
        n_stages = int(_weighted(rng, N_STAGES_DIST))
        schema = {"row_count":row_count,"columns":["c%d" % i for i in range(rng.randint(*COLUMN_RANGE))],"size_hint":tier}
        for idx in range(n_stages):
            stage = {"type":"copy"} if idx == 0 else make_notebook_stage(rng, idx == n_stages-1)
            feat = stage_features(stage, schema, csv_size_bytes, idx, n_stages)
            fl = dict(feat); fl["row_count"] = int(feat["row_count"]*intensity); fl["csv_size_mb"] = feat["csv_size_mb"]*intensity
            lab = label_settings(fl, rng)
            rows.append(dict(list({c:feat[c] for c in FEATURE_COLS}.items())
                             + list({c:lab[c] for c in TARGET_COLS}.items())
                             + [(NODE_TARGET, lab[NODE_TARGET])]))
            if len(rows) >= n_rows: break
    return pd.DataFrame(rows)

df = generate(N_ROWS, SEED)
print("generated:", df.shape)
df.to_csv("resource_training.csv", index=False)
df.head()

### Quick sanity - labels should behave sensibly
Workers rise with size, aggregations pull more workers, DIU spans 2-8.

In [ ]:
nb = df[df.stage_is_copy == 0]
print("workers dist      :", nb.rec_workers.value_counts().sort_index().to_dict())
print("workers by size   :", nb.groupby("size_hint_ord").rec_workers.mean().round(2).to_dict())
print("agg -> workers    :", nb.groupby("has_aggregation").rec_workers.mean().round(2).to_dict())
print("node dist         :", nb.rec_node_type.value_counts().to_dict())
print("DIU dist (copy)   :", df[df.stage_is_copy == 1].rec_diu.value_counts().sort_index().to_dict())

## 4. Train - 4 regressors + 1 node classifier

Multi-target `HistGradientBoosting` (fast + accurate on 500k tabular rows). Fits on plain
numpy so the FastAPI process (which passes numpy vectors) gets no feature-name warnings.

In [ ]:
import numpy as np, json
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, r2_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLS].values
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
Xtr, Xte = X[idx_tr], X[idx_te]

regressors, metrics = {}, {"sklearn_version": sklearn.__version__, "rows": int(len(df)), "targets": {}}
for tgt in TARGET_COLS:
    y = df[tgt].values
    reg = HistGradientBoostingRegressor(max_iter=400, max_depth=8, learning_rate=0.06,
                                        l2_regularization=1.0, random_state=42)
    reg.fit(Xtr, y[idx_tr])
    pred, yte = reg.predict(Xte), y[idx_te]
    entry = {"mae": round(float(mean_absolute_error(yte, pred)), 3),
             "r2": round(float(r2_score(yte, pred)), 4)}
    if tgt in ("rec_workers", "rec_diu"):
        rp = np.clip(np.round(pred), 0, None)
        entry["within1_acc"] = round(float((np.abs(rp - yte) <= 1).mean()), 4)
    regressors[tgt] = reg; metrics["targets"][tgt] = entry
    print(tgt.ljust(24), "MAE=%-8s R2=%s  %s" % (entry["mae"], entry["r2"], entry.get("within1_acc","")))

node_clf = HistGradientBoostingClassifier(max_iter=300, max_depth=8, learning_rate=0.08, random_state=42)
node_y = df[NODE_TARGET].values
node_clf.fit(Xtr, node_y[idx_tr])
bal = balanced_accuracy_score(node_y[idx_te], node_clf.predict(Xte))
metrics["node_type"] = {"balanced_accuracy": round(float(bal), 4)}
print(NODE_TARGET.ljust(24), "balanced_accuracy=%.4f" % bal)

## 5. Save the bundle - download & drop into the repo

Save `resource_models.pkl` + `metrics.json`, then in Kaggle: **Output** tab -> download both,
and place them in `unified/resource_agent/models/`. The agent auto-loads the bundle on next
start (verify via `GET /api/resource/model-info`).

In [ ]:
import joblib
bundle = {
    "feature_cols": FEATURE_COLS, "targets": TARGET_COLS, "node_target": NODE_TARGET,
    "regressors": regressors, "node_classifier": node_clf,
    "node_classes": [str(c) for c in node_clf.classes_],
    "sklearn_version": sklearn.__version__, "trained_rows": int(len(df)),
}
joblib.dump(bundle, "resource_models.pkl")
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("saved resource_models.pkl + metrics.json")
print(json.dumps(metrics, indent=2))

## 6. Smoke-test the saved bundle

Confirm it loads and produces sensible, within-limit recommendations - exactly what the
agent's `ml_predictor.ResourceMLPredictor` does at inference.

In [ ]:
b = joblib.load("resource_models.pkl")
BOUNDS = {"rec_workers":(0,MAX_WORKERS),"rec_diu":(0,MAX_DIU),
          "rec_memory_gb":(2.0,MAX_TOTAL_MEM_GB),"rec_shuffle_partitions":(SHUFFLE_TIERS[0],SHUFFLE_TIERS[-1])}

def recommend(stage, schema, csv_size_bytes=0, stage_index=0, n_stages=1):
    feat = stage_features(stage, schema, csv_size_bytes, stage_index, n_stages)
    Xv = np.array([feat[c] for c in FEATURE_COLS], dtype=float).reshape(1, -1)
    r = b["regressors"]
    out = {"workers": int(np.clip(round(float(r["rec_workers"].predict(Xv)[0])), *BOUNDS["rec_workers"])),
           "diu": int(np.clip(round(float(r["rec_diu"].predict(Xv)[0])), *BOUNDS["rec_diu"])),
           "memory_gb": round(float(np.clip(r["rec_memory_gb"].predict(Xv)[0], *BOUNDS["rec_memory_gb"])), 2),
           "shuffle": snap_shuffle(float(np.clip(r["rec_shuffle_partitions"].predict(Xv)[0], *BOUNDS["rec_shuffle_partitions"]))),
           "node": str(b["node_classifier"].predict(Xv)[0])}
    if feat["stage_is_copy"]: out["workers"], out["node"] = 0, DEFAULT_NODE
    else: out["diu"] = 0
    return out

cases = [
    ("tiny copy",  {"type":"copy"}, {"row_count":2000,"columns":list("ab"),"size_hint":"small"}, 2000*140),
    ("big copy",   {"type":"copy"}, {"row_count":80_000_000,"columns":list("abcde"),"size_hint":"xlarge"}, 80_000_000*140),
    ("small nb",   {"type":"notebook","transformations":["x=1"]}, {"row_count":20_000,"columns":list("abc"),"size_hint":"small"}, 20_000*140),
    ("heavy agg",  {"type":"notebook","transformations":["x=1"],"aggregations":{"group_by":["g"],"agg_exprs":["sum(a)","avg(b)"]}}, {"row_count":40_000_000,"columns":list("abcdefgh"),"size_hint":"xlarge"}, 40_000_000*140),
]
for name, st, sc, sz in cases:
    print(name.ljust(10), recommend(st, sc, sz, 1, 3))